# 桶排序
按MA分MA_BUCKETS_NUM个桶，按SA分SA_BUCKETS_NUM个桶，然后按MA和SA的组合分组，构建MA_BUCKETS_NUM * SA_BUCKETS_NUM个组合。  




## 导入库

In [95]:
import warnings
from typing import Any
from pathlib import Path
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
import plotly.io as pio
import plotly.graph_objects as go
import plotly.subplots as sp
import numpy as np
import statsmodels.api as sm
from scipy.stats import t as t_dist

## 超参数 

In [96]:
import os
import dotenv
dotenv.load_dotenv()
CONNECTION_URL = os.getenv("POSTGRES_URL")
ENGINE = "adbc"

TASK_ID_PREFIX = 'baseline1'  # 任务id前缀
RESULTS_BASE_DIR = '/home/frank/files/programs/GraduationThesis/result' # 基本数据路径
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{TASK_ID_PREFIX}' # 保存基本路径
SAVE_BASELINE_REG_DIR = SAVE_BASE_DIR + '/baseline_reg'
SAVE = True # 是否保存数据

MA_BUCKETS_NUM = 3 # MA分桶数  
SA_BUCKETS_NUM = 3 # SA分桶数  
BUCKETS_NUM = MA_BUCKETS_NUM * SA_BUCKETS_NUM # 组合数

RISK_FREE_RATE = 0.015 / 12 # 无风险利率  

FILTER_BUCKET_MA_GROUP = True # 仅选取某一组
GROUP = 3 # 组号 1,2,3

## 读取数据  
需要读取masa数据和市值数据  
市值数据：数据库 or 本地json (测试用)   


In [97]:
masa_lf = pl.scan_parquet(SAVE_BASELINE_REG_DIR + '/基准回归-MA,SA双因子.parquet')   
# 市值数据从 DB 读取（statics.market_value）
market_value_lf = pl.read_database_uri(
    uri=CONNECTION_URL,
    query='SELECT stkcd AS "Stkcd", trdmnt AS "Trdmnt", msmvosd AS "Msmvosd" FROM statics.market_value',
    engine=ENGINE,
).lazy()

## 处理数据(测试)

In [98]:
market_value_lf = market_value_lf.rename(
    {
        'Stkcd':'portfolio',
        'Trdmnt':'date',
        'Msmvosd':'msmvosd'
    }
)

## 桶排序  
先按MA分桶，再按SA分桶，然后按MA和SA的组合分组，构建MA_BUCKETS_NUM * SA_BUCKETS_NUM个组合。  

具体来说，假设两者桶数都是3  
MA先排出ma_id 0,1,2  
然后SA遍历ma_id = 0,1,2 
里面排sa_id = MA_BUCKETS_NUM * ma_id + sa_id

两者都从小到大排（SA为标准差倒数，所以是从小到大排）  


In [99]:
# 桶号
masa_lf = masa_lf.with_columns(
    [
        pl.col('MA').quantile(i / MA_BUCKETS_NUM, interpolation='lower').alias(f'ma_q{i}')
        for i in range(1,MA_BUCKETS_NUM)
    ]
)
masa_lf = masa_lf.with_columns(
    [
        pl.col('SA').quantile(i / SA_BUCKETS_NUM, interpolation='lower').alias(f'sa_q{i}')
        for i in range(1,SA_BUCKETS_NUM)
    ]
)

# 连接市值
joined_lf = masa_lf.join(market_value_lf, on=['portfolio','date'], how='left')
joined_lf = joined_lf.with_columns(
    (pl.col('msmvosd') / pl.col('msmvosd').sum().over('date')).alias('weight')
)

ma_lf_list = list[pl.LazyFrame]() # 每一个桶对应的series，date-weighted_sum_ret
for i in range(0,MA_BUCKETS_NUM): #
    # 第1个组合
    if i == 0:
        bucket_lf = joined_lf.filter(pl.col(f'MA') <= pl.col(f'ma_q1'))
    elif i == (MA_BUCKETS_NUM - 1):
        bucket_lf = joined_lf.filter(pl.col(f'MA') > pl.col(f'ma_q{i}'))
    # 最后一个组合
    else:
        bucket_lf = joined_lf.filter((pl.col(f'MA') > pl.col(f'ma_q{i}')) & (pl.col(f'MA') <= pl.col(f'ma_q{i+1}')))
    bucket_lf = bucket_lf.with_columns(pl.lit(i).alias('ma_bucket_id'))
    bucket_lf = bucket_lf.drop(pl.selectors.starts_with('ma_q'),pl.col('MA'))
    
    # 添加到series_list
    ma_lf_list.append(bucket_lf)

sa_lf_list = list[pl.LazyFrame]()
for i in range(0,MA_BUCKETS_NUM):
    lf = ma_lf_list[i]
    for j in range(0,SA_BUCKETS_NUM):
        if j == 0:
            bucket_lf = lf.filter(pl.col(f'SA') <= pl.col(f'sa_q{1}'))
        elif j == (SA_BUCKETS_NUM - 1):
            bucket_lf = lf.filter(pl.col(f'SA') > pl.col(f'sa_q{j}'))
        else:
            bucket_lf = lf.filter((pl.col(f'SA') > pl.col(f'sa_q{j}')) & (pl.col(f'SA') <= pl.col(f'sa_q{j+1}')))
        bucket_lf = bucket_lf.with_columns(pl.lit(i * MA_BUCKETS_NUM + j).alias('bucket_id'))
        bucket_lf = bucket_lf.drop(pl.selectors.starts_with('sa_q'),pl.col('SA'),pl.col('ma_bucket_id'))
        sa_lf_list.append(bucket_lf)

# 合并
combined_series = pl.concat(sa_lf_list, how='vertical')

# 分组求市值
combined_series = combined_series.group_by(['date','bucket_id']).agg(
    (pl.col('weight') * pl.col('return')).sum().alias('weighted_sum_ret'),
)

# 根据MA组筛选
if FILTER_BUCKET_MA_GROUP and isinstance(GROUP, int):
    combined_series = combined_series.filter((pl.col('bucket_id') / MA_BUCKETS_NUM).cast(pl.Int32) == (GROUP - 1))
    min_num = combined_series.select('bucket_id').min().collect().item()
    max_num = combined_series.select('bucket_id').max().collect().item()
    print(f'MA组号为{GROUP}的组合号范围为{min_num}到{max_num}')
# 转bucker_id为str
combined_series = combined_series.with_columns(
    pl.col('bucket_id').cast(pl.Utf8).alias('bucket_id')
)


combined_series.head().collect()

MA组号为3的组合号范围为6到8


date,bucket_id,weighted_sum_ret
date,str,f64
2023-12-01,"""8""",-0.007612
2017-12-01,"""6""",-0.000496
2019-05-01,"""7""",0.000007
2022-08-01,"""7""",-0.001318
2012-06-01,"""8""",-0.003651


#### 对冲组合  
做多第BUCKET_NUM-1个组合，做空第1个组合。 

In [100]:
# 构建对冲组合
low_bucket = combined_series.filter(pl.col('bucket_id') == str(min_num)).rename({'weighted_sum_ret':'low_bucket_ret'})
high_bucket = combined_series.filter(pl.col('bucket_id') == str(max_num)).rename({'weighted_sum_ret':'high_bucket_ret'})  
joined_bucket = low_bucket.join(high_bucket, on='date', how='left')
joined_bucket = joined_bucket.with_columns(
    pl.col('high_bucket_ret').fill_null(0).alias('high_bucket_ret'),
    pl.col('low_bucket_ret').fill_null(0).alias('low_bucket_ret'),
)
joined_bucket = joined_bucket.with_columns(
    (pl.col('high_bucket_ret') - pl.col('low_bucket_ret')).alias('weighted_sum_ret')
)
joined_bucket = joined_bucket.select('date','weighted_sum_ret').with_columns(
    pl.lit('对冲组合').alias('bucket_id')
)

joined_bucket = joined_bucket.select(['date','bucket_id','weighted_sum_ret'])

joined_bucket.head().collect()

date,bucket_id,weighted_sum_ret
date,str,f64
2018-02-01,"""对冲组合""",0.002623
2009-05-01,"""对冲组合""",0.00206
2023-05-01,"""对冲组合""",0.001579
2008-03-01,"""对冲组合""",-0.001108
2012-12-01,"""对冲组合""",0.003909


In [101]:
combined_series = pl.concat([combined_series, joined_bucket], how='vertical').sort(['date','bucket_id'])
combined_series.head().collect()

date,bucket_id,weighted_sum_ret
date,str,f64
2004-12-01,"""6""",-0.000447
2004-12-01,"""7""",-0.00134
2004-12-01,"""8""",-0.004247
2004-12-01,"""对冲组合""",-0.0038
2005-01-01,"""6""",0.001168


In [102]:
if SAVE:
    combined_series.collect().write_parquet(SAVE_BASELINE_REG_DIR + '/基准回归-分桶市值加权收益.parquet')

### 平均收益
#### 每个桶的平均收益（包括对冲组合）    
对于每个桶序列，计算其各个时期的平均收益, HAC-t 和 p  

计算方式为，使用`weighted_sum_ret`回归常数项，使用HAC-t和p。  

In [103]:
# 每个 bucket：weighted_sum_ret 的 mean、HAC-t、p（只回归常数项，无 market_ret）

coll = combined_series.collect()
def regress_one(s: pl.DataFrame) -> pl.DataFrame:
    g = s.to_pandas()
    bid = g['bucket_id'].iloc[0]
    try:
        y = g['weighted_sum_ret'].to_numpy()
        x = np.ones((len(y), 1))
        results = sm.OLS(y, x).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        return pl.DataFrame({
            'bucket_id': [bid],
            'mean_return': [float(results.params[0])],
            't': [float(results.tvalues[0])],
            'p': [float(results.pvalues[0])],
        })
    except Exception as e:
        warnings.warn(f'计算{bid}时发生错误: {e}')
        return pl.DataFrame({
            'bucket_id': [bid],
            'mean_return': [0.0],
            't': [0.0],
            'p': [1.0],
        })
mean_tp_table = coll.group_by('bucket_id').map_groups(regress_one)

In [104]:
# 格式化：4 位有效数字
mean_tp_table = mean_tp_table.with_columns(
    pl.col('mean_return').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('mean_return'),
    pl.col('t').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('p'),
)
# t、p 加括号
mean_tp_table = mean_tp_table.select(
    pl.col('bucket_id'),
    pl.col('mean_return'),
    (pl.lit('[') + pl.col('t') + pl.lit(']')).alias('t'),
    (pl.lit('(') + pl.col('p') + pl.lit(')')).alias('p'),
)
# 居中对齐到 12 位
mean_tp_table = mean_tp_table.with_columns(
    pl.col('mean_return').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('mean_return'),
    pl.col('t').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('p'),
)
# 合并为一行展示
mean_tp_table = mean_tp_table.select(
    pl.col('bucket_id'),
    (pl.col('mean_return') + pl.lit('\n') + pl.col('t') + pl.lit('\n') + pl.col('p')).alias('mean_return'),
).sort('bucket_id')

with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(mean_tp_table)

bucket_id,mean_return
str,str
"""6""",""" 0.000173 [2.209] (0.0272) """
"""7""",""" 0.0008824 [3.019] (0.002534) """
"""8""",""" 0.002008 [3.638] (0.0002744) """
"""对冲组合""",""" 0.001835 [3.81] (0.0001391) """


In [105]:
if SAVE:
    mean_tp_table.write_parquet(SAVE_BASELINE_REG_DIR + '/基准回归-分桶均值HACtp.parquet')

将对冲组合的收益序列添加到桶排序的底部  

绘制panel_list的累计收益曲线  

绘制方法为，对于每一个时序数据，计算每一期累计收益，然后绘制成曲线。  

In [106]:
# 累加
combined_series = combined_series.with_columns(pl.col('weighted_sum_ret').cum_sum().over('bucket_id').alias('cum_return'))

# 百分化
combined_series = combined_series.with_columns(
    pl.col('cum_return').mul(100).alias('cum_return')
)

# 平滑
ROLLING_WINDOW = 4
MIN_SAMPLES = 1
combined_series = combined_series.with_columns(
    pl.col('cum_return').rolling_mean(window_size=ROLLING_WINDOW, min_samples=MIN_SAMPLES).over('bucket_id').alias('cum_return_smooth')
)

# 绘图
fig = px.line(combined_series.collect(), x='date', y='cum_return_smooth', color='bucket_id')
fig.update_layout(
    title=f'累计收益按桶分组(平滑窗口={ROLLING_WINDOW},最小样本={MIN_SAMPLES})',           # 图标题
    xaxis_title='日期',                # x 轴名称
    yaxis_title=f'累计收益率(%)',           # y 轴名称
)
fig.show()
    
    

In [107]:
if SAVE:
    fig.write_image(SAVE_BASELINE_REG_DIR + '/基准回归-累计收益率分桶.png')

### Sharp  
计算每个组合的Sharp比率    

计算方式为：$(mean(ret) - RISK\_FREE\_RATE) / std(ret)$


In [108]:
sharp_series = combined_series.group_by(['bucket_id']).agg(
    pl.col('weighted_sum_ret').mean().alias('mean_ret'),
    pl.col('weighted_sum_ret').std().alias('std_ret'),
)

sharp_series = sharp_series.select(
    pl.col('bucket_id'),
    ((pl.col('mean_ret') - RISK_FREE_RATE) / pl.col('std_ret')).alias('sharp')
)

sharp_series = sharp_series.with_columns(
    pl.col('sharp').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('sharp'),
)
sharp_series = sharp_series.sort('bucket_id')
sharp_series = sharp_series.collect()
sharp_series

bucket_id,sharp
str,str
"""6""","""-1.006"""
"""7""","""-0.1001"""
"""8""","""0.1088"""
"""对冲组合""","""0.0967"""


### 合并+保存 收益和sharp

In [109]:
performance = mean_tp_table.join(sharp_series, on='bucket_id', how='left').sort('bucket_id')

if SAVE:
    performance.write_parquet(SAVE_BASELINE_REG_DIR + '/基准回归-分桶表现.parquet')